In [1]:
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import joblib

# add path to project root
import sys
PROJECT_ROOT = Path.cwd()

if "notebooks" in PROJECT_ROOT.parts:
    PROJECT_ROOT = PROJECT_ROOT.parent
    
sys.path.append(str(PROJECT_ROOT))


from src.preprocess.gaming_text_preprocessor import GamingTextPreprocessor
from src.preprocess.gaming_text_labeler import GamingTextLabeler
from src.ensemble.ensemble import Ensemble
from src.model.bert_collection import BertToxicityModelCollection
from src.ensemble.score import ClassificationMetrics

# instantiate preprocessor and labeler
gaming_labeler = GamingTextLabeler()
metrics = ClassificationMetrics()



In [2]:
CONFIG = {
    'seed': 7524,
    'data_dir': PROJECT_ROOT / 'data' / 'processed_data',
    'model_dir': PROJECT_ROOT / 'models',
    'text_col': 'message',
    'label_col': 'label'
}

np.random.seed(CONFIG['seed'])
seed = CONFIG['seed']
tc, lc = CONFIG['text_col'], CONFIG['label_col']

print('CONFIG loaded. Text column:', tc)

CONFIG loaded. Text column: message


In [3]:
# load WOT train data
d = CONFIG['data_dir']

wot_train = pd.read_parquet(d / 'wot' / 'x_train.parquet')
wot_train["data_source"] = "wot"

# load DOTA training data
dota_train = pd.read_parquet(d / 'dota' / 'x_train.parquet')
dota_train["data_source"] = "dota"

# combine datasets
train = pd.concat([wot_train, dota_train], ignore_index=True)

# clean labels and create 3-class label column
train = gaming_labeler.convert_three_class(
    train, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)

In [4]:
wot_val = pd.read_parquet(d / 'wot' / 'x_validation.parquet')
wot_val["data_source"] = "wot"
dota_val = pd.read_parquet(d / 'dota' / 'x_validation.parquet')
dota_val["data_source"] = "dota"

# combine holdout datasets
val = pd.concat([wot_val, dota_val], ignore_index=True)

# clean labels and create 3-class label column
val = gaming_labeler.convert_three_class(
    val, 
    label_column=lc, 
    data_source_column='data_source',
    output_column='label_3class'
)

# Create Objects 

In [5]:
bert_multi_model = BertToxicityModelCollection(
    model_names=["dota_multi", "jigsaw_multi", "wot_multi"],
)

Loading dota_multi from jforward/bert-toxicity/dota_multi_model...
Loading jigsaw_multi from jforward/bert-toxicity/jigsaw_multi_model...
Loading wot_multi from jforward/bert-toxicity/wot_multi_model...


In [6]:
bert_multi_ensemble = Ensemble(
    model_collections=[bert_multi_model]
)

# Simple Majority Rules

In [7]:
pred_train = bert_multi_ensemble.predict_simple_majority(train[tc])
metrics.metrics(train['label_3class'], pred_train)

Predicting with SimpleMajority...
Input has 53707 samples.
dota_multi: (53707,)
jigsaw_multi: (53707,)
wot_multi: (53707,)
Collected predictions from 3 models.
Prediction matrix shape: (53707, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.5327168007618174,
 'CV Weighted F1': 0.7743964328367929,
 'Accuracy': 0.8191669614761576,
 'Coverage': 1.0,
 'Precision': 0.7113251507832531}

In [8]:
pred_train = bert_multi_ensemble.predict_simple_majority(val[tc])
metrics.metrics(val['label_3class'], pred_train)

Predicting with SimpleMajority...
Input has 17056 samples.
dota_multi: (17056,)
jigsaw_multi: (17056,)
wot_multi: (17056,)
Collected predictions from 3 models.
Prediction matrix shape: (17056, 3)
Aggregating predictions from 3 models...


{'CV Macro F1': 0.4990977903561413,
 'CV Weighted F1': 0.7724849682856554,
 'Accuracy': 0.8178353658536586,
 'Coverage': 1.0,
 'Precision': 0.6380795925531287}

# Confidene

In [10]:
thresholds = np.array([0.5, 0.9, 0.001])

res = bert_multi_ensemble.fit_weighted_confidence_majority(
    X_val=train[tc], 
    y_val=train['label_3class'], 
    score_func=metrics.score,
    thresholds=thresholds
)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=53707, n_classes=3)
Starting random search for n models: 3
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape: (53707, 3)
Trial weights: (3,)
Weighted probabilities shape

In [11]:
combined_pred = bert_multi_ensemble.predict_weighted_confidence_majority(
    X=train[tc],
    weights=res[0],
    threshold=res[1]
)
metrics.metrics(train['label_3class'], combined_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=53707, n_classes=3)
Predicted labels shape: (53707,)


{'CV Macro F1': 0.6695288529479432,
 'CV Weighted F1': 0.8481022478825977,
 'Accuracy': 0.8670378162995512,
 'Coverage': 1.0,
 'Precision': 0.7689648577129958}

In [12]:
combined_pred = bert_multi_ensemble.predict_weighted_confidence_majority(
    X=val[tc],
    weights=res[0],
    threshold=res[1]
)
metrics.metrics(val['label_3class'], combined_pred)

Constructing confidence tensor...
Total models in ensemble: 3
Expected confidence shape: (n_samples=17056, n_classes=3)
Predicted labels shape: (17056,)


{'CV Macro F1': 0.6195429228955077,
 'CV Weighted F1': 0.8283429647903731,
 'Accuracy': 0.8465056285178236,
 'Coverage': 1.0,
 'Precision': 0.690274291641266}

In [13]:
res

(array([0.51393845, 0.00572898, 0.48033257]),
 0.001,
 0.6695288529479432,
 ['dota_multi', 'jigsaw_multi', 'wot_multi'])